#  Notebook 4 — Visualization & Inference (Faster R-CNN)


In [ ]:
# ============================================================
#  CONFIG
# ============================================================
import os

DATASET_BASE    = "dataset"
MODEL_LOAD_PATH = "saved_model/fasterrcnn_egypt.pth"  
OUTPUT_DIR      = "output_images"                     
IMG_SIZE        = 800    # same as training
CONF_THRESH     = 0.5    # confidence threshold
NMS_THRESH      = 0.45   # NMS IoU threshold
NUM_VIZ         = 5      # side-by-side samples
GRID_NUM        = 9      # grid samples
GRID_COLS       = 3      
CUSTOM_IMAGE    = "our_image.jpg" 

CLASSES = [
    "traffic light", "traffic sign", "car", "person", "bus",
    "truck", "rider", "bike", "motor", "train", "banner", "tuktuk"
]
NUM_CLASSES = len(CLASSES) + 1
COLORS = [
    (255,0,0),(0,200,0),(0,0,255),(255,200,0),(200,0,200),(0,200,200),
    (180,0,0),(0,130,0),(0,0,180),(180,180,0),(180,0,180),(0,180,180)
]
SPLITS = {
    "val" : os.path.join(DATASET_BASE, "val",  "val"),
    "test": os.path.join(DATASET_BASE, "test", "test"),
}
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Config ready!")

## Cell 1 — Load Model

In [ ]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_LOAD_PATH, map_location=DEVICE))
model.to(DEVICE); model.eval()
print(f" Model loaded from: {MODEL_LOAD_PATH}")

## Cell 2 — Inference Helper

In [ ]:
import cv2, torch
import numpy as np
import torchvision.transforms as T
from torchvision.ops import nms

def run_inference(image_path, conf=CONF_THRESH, nms_thresh=NMS_THRESH):
    image_bgr = cv2.imread(image_path)
    if image_bgr is None: return None, None, 0
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w      = image_bgr.shape[:2]

    img_resized = cv2.resize(image_rgb, (IMG_SIZE, IMG_SIZE))
    tensor      = T.ToTensor()(img_resized).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(tensor)[0]

    # Filter by confidence
    mask   = pred['scores'] >= conf
    boxes  = pred['boxes'][mask]
    labels = pred['labels'][mask]
    scores = pred['scores'][mask]

    # Apply NMS
    if len(boxes) > 0:
        keep   = nms(boxes, scores, nms_thresh)
        boxes  = boxes[keep].cpu().numpy()
        labels = labels[keep].cpu().numpy()
        scores = scores[keep].cpu().numpy()
        # Scale boxes back to original image size
        boxes[:, [0,2]] = boxes[:, [0,2]] * w / IMG_SIZE
        boxes[:, [1,3]] = boxes[:, [1,3]] * h / IMG_SIZE
    else:
        boxes = labels = scores = np.array([])

    # Draw predictions
    drawn = image_rgb.copy()
    for box, label, score in zip(boxes, labels, scores):
        cls_id = int(label) - 1
        if cls_id < 0 or cls_id >= len(CLASSES): continue
        box    = box.astype(int)
        color  = COLORS[cls_id % len(COLORS)]
        cv2.rectangle(drawn, (box[0],box[1]), (box[2],box[3]), color, 2)
        cv2.putText(drawn, f"{CLASSES[cls_id]} {score:.2f}",
                    (box[0], max(box[1]-5,10)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return image_rgb, drawn, len(boxes) if hasattr(boxes,'__len__') else 0

def draw_gt(image_path, label_path):
    image = cv2.imread(image_path)
    if image is None: return None
    vis = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).copy()
    h, w = image.shape[:2]
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5: continue
                cls_id = int(parts[0])
                cx,cy,bw,bh = map(float, parts[1:])
                x1 = int((cx-bw/2)*w); y1 = int((cy-bh/2)*h)
                x2 = int((cx+bw/2)*w); y2 = int((cy+bh/2)*h)
                color = COLORS[cls_id % len(COLORS)]
                cv2.rectangle(vis,(x1,y1),(x2,y2),color,2)
                cv2.putText(vis,CLASSES[cls_id],(x1,max(y1-5,10)),
                            cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)
    return vis

print("Helper functions ready!")

## Cell 3 — GT vs Prediction Side-by-Side

In [ ]:
import random, matplotlib.pyplot as plt

VIZ_SPLIT = "val" 
img_dir = os.path.join(SPLITS[VIZ_SPLIT], "images")
lbl_dir = os.path.join(SPLITS[VIZ_SPLIT], "labels")
imgs    = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
samples = random.sample(imgs, min(NUM_VIZ, len(imgs)))

for img_file in samples:
    img_path = os.path.join(img_dir, img_file)
    lbl_path = os.path.join(lbl_dir, os.path.splitext(img_file)[0]+".txt")
    gt_img   = draw_gt(img_path, lbl_path)
    orig, pred_img, n_det = run_inference(img_path)
    if gt_img is None or pred_img is None: continue
    n_gt = sum(1 for _ in open(lbl_path)) if os.path.exists(lbl_path) else 0

    fig, axes = plt.subplots(1,2,figsize=(18,7))
    axes[0].imshow(gt_img);   axes[0].set_title(f"Ground Truth ({n_gt} objects)",fontsize=13)
    axes[1].imshow(pred_img); axes[1].set_title(f"Faster R-CNN ({n_det} detections, conf≥{CONF_THRESH})",fontsize=13)
    for ax in axes: ax.axis('off')
    plt.suptitle(img_file, fontsize=9, color='gray')
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, f"viz_{img_file}")
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"GT: {n_gt} | Pred: {n_det} | Saved: {save_path}")

## Cell 4 — Prediction Grid

In [ ]:
import random, math, cv2
import matplotlib.pyplot as plt
import numpy as np

THUMB_SIZE  = (400,400)  
GRID_SPLIT  = "test"     

img_dir = os.path.join(SPLITS[GRID_SPLIT], "images")
imgs    = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
samples = random.sample(imgs, min(GRID_NUM, len(imgs)))
frames  = []
for img_file in samples:
    _, pred_img, _ = run_inference(os.path.join(img_dir, img_file))
    if pred_img is not None:
        frames.append(cv2.resize(pred_img, THUMB_SIZE))

if frames:
    rows = math.ceil(len(frames)/GRID_COLS)
    fig, axes = plt.subplots(rows, GRID_COLS, figsize=(GRID_COLS*4, rows*4))
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < len(frames): ax.imshow(frames[i]); ax.axis('off')
        else: ax.axis('off')
    plt.suptitle(f"Faster R-CNN Prediction Grid (conf≥{CONF_THRESH})",fontsize=13)
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR,"prediction_grid.png")
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f" Saved: {save_path}")

## Cell 5 — Custom Image Inference

In [ ]:
import matplotlib.pyplot as plt

if not os.path.exists(CUSTOM_IMAGE):
    print(f" Not found: '{CUSTOM_IMAGE}'\n   Set CUSTOM_IMAGE in CONFIG cell.")
else:
    orig, result_img, n_det = run_inference(CUSTOM_IMAGE, conf=CONF_THRESH)
    plt.figure(figsize=(13,7))
    plt.imshow(result_img)
    plt.title(f"{n_det} objects detected (conf≥{CONF_THRESH})")
    plt.axis('off'); plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR,"custom_inference.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f" {n_det} objects found | Saved: {save_path}")

## Cell 6 — Compare Pretrained vs Fine-tuned

In [ ]:
import random, matplotlib.pyplot as plt
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights

# Load baseline (pretrained COCO, no fine-tuning)
baseline = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
baseline.to(DEVICE); baseline.eval()
print(" Baseline (pretrained COCO) loaded")

NUM_COMPARE = 3  
img_dir     = os.path.join(SPLITS["val"], "images")
lbl_dir     = os.path.join(SPLITS["val"], "labels")
imgs        = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
samples     = random.sample(imgs, min(NUM_COMPARE, len(imgs)))

for img_file in samples:
    img_path = os.path.join(img_dir, img_file)
    lbl_path = os.path.join(lbl_dir, os.path.splitext(img_file)[0]+".txt")
    gt_img   = draw_gt(img_path, lbl_path)

    # Fine-tuned model
    _, ft_img, n_ft = run_inference(img_path)

    # Baseline model — temporarily swap
    orig_model = model
    import types
    # run baseline inference directly
    import torchvision.transforms as T
    image_bgr = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w = image_bgr.shape[:2]
    tensor = T.ToTensor()(cv2.resize(image_rgb,(IMG_SIZE,IMG_SIZE))).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        bp = baseline(tensor)[0]
    bmask = bp['scores'] >= CONF_THRESH
    bl_img = image_rgb.copy()
    for box, label, score in zip(bp['boxes'][bmask], bp['labels'][bmask], bp['scores'][bmask]):
        box = box.cpu().numpy().astype(int)
        box[[0,2]] = (box[[0,2]] * w / IMG_SIZE).astype(int)
        box[[1,3]] = (box[[1,3]] * h / IMG_SIZE).astype(int)
        cv2.rectangle(bl_img,(box[0],box[1]),(box[2],box[3]),(128,128,128),2)
        cv2.putText(bl_img,f"cls{int(label)} {float(score):.2f}",(box[0],max(box[1]-5,10)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(128,128,128),2)
    n_gt = sum(1 for _ in open(lbl_path)) if os.path.exists(lbl_path) else 0

    fig, axes = plt.subplots(1,3,figsize=(24,7))
    axes[0].imshow(gt_img);  axes[0].set_title(f"Ground Truth ({n_gt})",fontsize=12)
    axes[1].imshow(bl_img);  axes[1].set_title(f"Baseline COCO ({bmask.sum()} det)",fontsize=12)
    axes[2].imshow(ft_img);  axes[2].set_title(f"Fine-tuned ({n_ft} det)",fontsize=12)
    for ax in axes: ax.axis('off')
    plt.suptitle(img_file, fontsize=9, color='gray')
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR,f"compare_{img_file}")
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()